In [21]:
import pandas as pd
import numpy as np
data_dir = "../data/"
data_path = "/orcd/pool/003/dbertsim_shared/ukb/"

In [26]:
CANCER_COLS = ["breast","ovarian","uterine"]
# CANCER_COLS = ["prostate"]
cancer_cols = [f"{cancer}_cancer" for cancer in CANCER_COLS]
time_cols = [f"{cancer}_cancer_time_to_diagnosis" for cancer in CANCER_COLS]

In [28]:
df = pd.read_csv(f"{data_path}blood_protein_cancers_clean.csv")
df = df.drop(columns = [c for c in df.columns if "cancer" in c and c not in cancer_cols and c not in time_cols])
df = df.loc[df['Sex_male'] == 0]

In [29]:
df.head()

,eid,Age at recruitment,Sex_male,Ethnic background,Body mass index (BMI),"Systolic blood pressure, automated reading","Diastolic blood pressure, automated reading",Townsend deprivation index at recruitment,Smoking status,Alcohol intake frequency.,...,blood_Triglycerides,blood_Urate,blood_Urea,blood_Vitamin D,breast_cancer,ovarian_cancer,uterine_cancer,breast_cancer_time_to_diagnosis,ovarian_cancer_time_to_diagnosis,uterine_cancer_time_to_diagnosis
0,1000083,49,0,British,24.7295,116.0,71.0,-3.96,Previous,Three or four times a week,...,1.238,196.9,6.40,35.5,0,0,0,NaN,NaN,NaN
1,1000380,62,0,British,31.2026,124.0,81.0,-5.00,Never,Daily or almost daily,...,1.732,305.0,6.22,56.3,0,0,0,NaN,NaN,NaN
2,1001803,47,0,Any other white background,24.2187,98.0,57.0,2.00,Never,Never,...,1.361,260.2,4.24,36.7,0,0,0,NaN,NaN,NaN
4,1003287,69,0,British,28.1479,166.0,61.0,6.38,Previous,Three or four times a week,...,1.356,235.0,5.89,69.8,0,0,0,NaN,NaN,NaN
5,1003630,67,0,British,32.1094,154.0,85.0,3.52,Never,Never,...,0.852,334.1,4.92,33.0,0,0,0,NaN,NaN,NaN


## Train/val/test split of cancer dataset
1. Drop protein columns that have over 50% missing
2. Split into train and test using iterative-stratification on cancer + top 7 cancers (with each time frame)

In [30]:
def bin_ttd(x):
    if pd.isna(x):         return "NA"
    if x <= 30/365.25:     return "<0"
    if 30/365.25 < x <= 5: return "0-5"   # (0,1]
    return ">5"

def proportions(frame, label):
    return (frame[f"{label}_strata"].value_counts(normalize=False)
            .reindex(["<0","0-5",">5", "NA"])
            .fillna(0))

In [31]:
for cancer in CANCER_COLS:
    df[f"{cancer}_strata"] = df[f"{cancer}_cancer_time_to_diagnosis"].apply(bin_ttd)
    print(f"\n{cancer}:\n", proportions(df, cancer))


breast:
 <0       870
0-5      411
>5       593
NA     26693
Name: breast_strata, dtype: int64

ovarian:
 <0        66
0-5       60
>5        59
NA     28382
Name: ovarian_strata, dtype: int64

uterine:
 <0       117
0-5       65
>5       114
NA     28271
Name: uterine_strata, dtype: int64


In [32]:
## Create train, validation, and test datasets for predicting current cancer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
# Sanity check: compare prevalences per label in full vs train vs test
def prevalence(table, cols):
    return pd.DataFrame({
        "prevalence": [table[c].mean() for c in cols],
        "n": [table[c].sum() for c in cols]
    }, index=cols)

def multilabel_stratified_split(
    df: pd.DataFrame,
    test_size=0.4,
    random_state=42,
    time=0
):
    """
    Splits df into train/test so that each label in label_cols
    has (approximately) the same prevalence in both splits,
    accounting for multi-label rows.
    """
    # 1) Build the multi-label target matrix (n_samples x n_labels)
    # Y = df[label_cols].astype(int).to_numpy()
    df_copy = df.copy()
    Y_cols = []
    for label in CANCER_COLS:
        for strata in ["<0","0-5",">5"]:
            df_copy[f"{label}_time_to_diagnosis_{strata}"] = (df[f"{label}_strata"] == strata)
            Y_cols.append(f"{label}_time_to_diagnosis_{strata}")
    Y = df_copy[Y_cols].to_numpy()

    # 2) Set up the multi-label stratified splitter
    msss = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )

    # 3) Run the split; indices refer to rows of df
    (train_idx, test_idx), = msss.split(df, Y)
    
    train_df = df.iloc[train_idx].copy()
    test_df  = df.iloc[test_idx].copy()

    train_df_copy = df_copy.iloc[train_idx].copy()
    test_df_copy  = df_copy.iloc[test_idx].copy()
    
    summary = pd.concat(
        {
            "train": prevalence(train_df_copy, Y_cols),
            "test": prevalence(test_df_copy, Y_cols),
        },
        axis=1,
    )
    print(summary)

    return train_df, test_df

# Make the split
train_df, validtest_df = multilabel_stratified_split(df, test_size=0.4)
valid_df, test_df = multilabel_stratified_split(validtest_df, test_size=0.5)
    
# train_df.to_csv(f"{data_path}ukb_cancer_train.csv", index=False)
# valid_df.to_csv(f"{data_path}ukb_cancer_valid.csv", index=False)
# test_df.to_csv(f"{data_path}ukb_cancer_test.csv", index=False)

                                   train            test     
                              prevalence    n prevalence    n
breast_time_to_diagnosis_<0     0.030455  522   0.030454  348
breast_time_to_diagnosis_0-5    0.014411  247   0.014352  164
breast_time_to_diagnosis_>5     0.020770  356   0.020740  237
ovarian_time_to_diagnosis_<0    0.002334   40   0.002275   26
ovarian_time_to_diagnosis_0-5   0.002100   36   0.002100   24
ovarian_time_to_diagnosis_>5    0.002042   35   0.002100   24
uterine_time_to_diagnosis_<0    0.004084   70   0.004113   47
uterine_time_to_diagnosis_0-5   0.002275   39   0.002275   26
uterine_time_to_diagnosis_>5    0.003967   68   0.004026   46
                                   train            test     
                              prevalence    n prevalence    n
breast_time_to_diagnosis_<0     0.030457  174   0.030452  174
breast_time_to_diagnosis_0-5    0.014353   82   0.014351   82
breast_time_to_diagnosis_>5     0.020655  118   0.020826  119
ovarian_

In [33]:
df_train = train_df
df_valid = valid_df
df_test = test_df

## Add follow up dates

In [35]:
data_path = "/orcd/pool/003/dbertsim_shared/ukb"

df_lab_time = rename_columns(pd.read_csv(f"../data/time_stamps_participant.csv"),field_dict)
df_assessment_centre = rename_columns(pd.read_csv(f"../data/assessment_centre.csv"),field_dict)
df = pd.merge(df_lab_time, df_assessment_centre, how = 'left', on = 'eid')

ASSESSMENT_CENTRE_TO_COUNTRY = {
    # England
    "Barts": "England",
    "Birmingham": "England",
    "Bristol": "England",
    "Bury": "England",
    "Cheadle (revisit)": "England",
    "Croydon": "England",
    "Hounslow": "England",
    "Leeds": "England",
    "Liverpool": "England",
    "Manchester": "England",
    "Middlesborough": "England",
    "Newcastle": "England",
    "Nottingham": "England",
    "Oxford": "England",
    "Reading": "England",
    "Sheffield": "England",
    "Stockport (pilot)": "England",
    "Stoke": "England",
    "Cheadle (imaging)": "England",
    "Reading (imaging)": "England",
    "Newcastle (imaging)": "England",
    "Bristol (imaging)": "England",

    # Scotland
    "Edinburgh": "Scotland",
    "Glasgow": "Scotland",

    # Wales
    "Cardiff": "Wales",
    "Swansea": "Wales",
    "Wrexham": "Wales",
}
COUNTRY_CENSOR_DATE = {
    "England":  pd.Timestamp("2023-05-31"),
    "Scotland": pd.Timestamp("2023-09-30"),
    "Wales":    pd.Timestamp("2016-12-31"),
}
df["country"] = df["UK Biobank assessment centre"].map(ASSESSMENT_CENTRE_TO_COUNTRY)
df["admin_censor_date"] = df["country"].map(COUNTRY_CENSOR_DATE)
df['admin_censor_date'] = pd.to_datetime(df['admin_censor_date'])

# if nan, then assume earlier date (2016)
df.loc[df['admin_censor_date'].isna(), 'admin_censor_date'] = pd.Timestamp("2016-12-31")
df['Date of attending assessment centre'] = pd.to_datetime(df['Date of attending assessment centre'])
df['time_to_follow_up'] = (df["admin_censor_date"] - df["Date of attending assessment centre"]).dt.days / 365.25

In [36]:
df_train = pd.merge(df_train, df[['eid','time_to_follow_up']], how = 'left', on = 'eid')
df_valid = pd.merge(df_valid, df[['eid','time_to_follow_up']], how = 'left', on = 'eid')
df_test = pd.merge(df_test, df[['eid','time_to_follow_up']], how = 'left', on = 'eid')

df_train = df_train.drop(columns = [col for col in df_train.columns if "strata" in col])
df_valid = df_valid.drop(columns = [col for col in df_valid.columns if "strata" in col])
df_test = df_test.drop(columns = [col for col in df_test.columns if "strata" in col])

df_train.to_csv(f"{data_path}/ukb_cancer_train_female.csv", index=False)
df_valid.to_csv(f"{data_path}/ukb_cancer_valid_female.csv", index=False)
df_test.to_csv(f"{data_path}/ukb_cancer_test_female.csv", index=False)

In [37]:
df_train.head()

,eid,Age at recruitment,Sex_male,Ethnic background,Body mass index (BMI),"Systolic blood pressure, automated reading","Diastolic blood pressure, automated reading",Townsend deprivation index at recruitment,Smoking status,Alcohol intake frequency.,...,blood_Urate,blood_Urea,blood_Vitamin D,breast_cancer,ovarian_cancer,uterine_cancer,breast_cancer_time_to_diagnosis,ovarian_cancer_time_to_diagnosis,uterine_cancer_time_to_diagnosis,time_to_follow_up
0,1000083,49,0,British,24.7295,116.0,71.0,-3.96,Previous,Three or four times a week,...,196.9,6.40,35.5,0,0,0,NaN,NaN,NaN,14.978782
1,1001803,47,0,Any other white background,24.2187,98.0,57.0,2.00,Never,Never,...,260.2,4.24,36.7,0,0,0,NaN,NaN,NaN,13.990418
2,1003287,69,0,British,28.1479,166.0,61.0,6.38,Previous,Three or four times a week,...,235.0,5.89,69.8,0,0,0,NaN,NaN,NaN,13.837098
3,1004330,57,0,Other ethnic group,26.1011,108.0,67.0,-2.66,Never,Never,...,256.1,5.53,66.7,0,0,0,NaN,NaN,NaN,14.817248
4,1005250,52,0,British,20.5859,159.0,89.0,-5.24,Never,Special occasions only,...,277.2,5.88,34.6,1,0,0,-1.03217,NaN,NaN,8.919918
